In [5]:
import pandas as pd
import numpy as np
import math
import operator
import os
import seaborn as sns

In [6]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.datasets import load_digits
from sklearn.metrics import pairwise_distances
from sklearn.manifold import TSNE
from sentence_transformers import SentenceTransformer
from keybert import KeyBERT
from keybert._mmr import mmr
from keybert._maxsum import max_sum_distance
from keybert._highlight import highlight_document
from keybert.backend._utils import select_backend
from tqdm import tqdm
from typing import List, Union, Tuple
from matplotlib import pyplot as plt
from scipy import linalg
from scipy.spatial.distance import squareform
from scipy.spatial.distance import pdist

/Users/jessie_guo/miniconda3/envs/Python3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
model = SentenceTransformer('all-mpnet-base-v2')

Loading weights: 100%|█| 199/199 [00:00<00:00, 10618.49it/s
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
pd.options.display.max_colwidth = 250
texts_path = '../Georgian Texts/Vazha Pshavela/'
poems_path = texts_path + 'Poems/revised/'

In [11]:
poem_names = []
sources = ['ge/', 'en/', 'ggl/', 'gem/', 'gem_ru/', 'gpt/', 'gpt_ru/']
ge_poems_directory = os.fsencode(poems_path + 'ge/')
for file in sorted(os.listdir(ge_poems_directory)):
    file = file.decode()
    if file.endswith(".md"):
        poem_names.append(file)
print(poem_names)

['01_i_believe_i_always_have_believed.md', '02_bakur.md', '03_some_day_it_will_happen_i_shall_die.md', '04_i_feel_like_singing_and_i_shall_sing.md', '05_voice_from_the_grave.md', '06_that_in_truth_is_not_manliness.md', '07_the_law_of_the_world_is_thus.md', '08_amiran.md', '09_the_old_song_of_cavaliers.md', '10_consciences_song.md', '11_as_once_you_did_o_lady_as_once_you_did.md', '12_loneliness.md', '13_what_created_me_a_human_being.md', '14_yet_again_shall_i_see_the_spring.md', '15_thrush_its_the_same_song_you_sing.md']


In [12]:
print('Number of verses')
print('==============================================================================================')
print('ID\tVazha\tHewitt\tGoogle\tGemini\tGem_ru\tChatGPT\tGPT_ru\tPassed?\tPoem Name')
for poem in poem_names:
    output = [poem[:2]]
    for source in sources:
        with open(poems_path + source + poem, "r") as file:
            output.append(str(sum(1 for _ in file)))
    if len(set(output[1:])) > 1:
        output.append('No')
    else:
        output.append('Yes')
    output.append(poem[3:-3].replace('_', ' ').title())
    print('\t'.join(output))

Number of verses
ID	Vazha	Hewitt	Google	Gemini	Gem_ru	ChatGPT	GPT_ru	Passed?	Poem Name
01	4	4	4	4	4	4	4	Yes	I Believe I Always Have Believed
02	36	36	36	36	36	36	36	Yes	Bakur
03	9	9	9	9	9	9	9	Yes	Some Day It Will Happen I Shall Die
04	14	14	14	14	14	14	14	Yes	I Feel Like Singing And I Shall Sing
05	18	18	18	18	18	18	18	Yes	Voice From The Grave
06	10	10	10	10	10	10	10	Yes	That In Truth Is Not Manliness
07	17	17	17	17	17	17	17	Yes	The Law Of The World Is Thus
08	32	32	32	32	32	32	32	Yes	Amiran
09	29	29	29	29	29	29	29	Yes	The Old Song Of Cavaliers
10	17	17	17	17	17	17	17	Yes	Consciences Song
11	12	12	12	12	12	12	12	Yes	As Once You Did O Lady As Once You Did
12	17	17	17	17	17	17	17	Yes	Loneliness
13	15	15	15	15	15	15	15	Yes	What Created Me A Human Being
14	27	27	27	27	27	27	27	Yes	Yet Again Shall I See The Spring
15	34	34	34	34	34	34	34	Yes	Thrush Its The Same Song You Sing


In [13]:
verses = []
poem_verse_counts = []
df = pd.DataFrame(
    columns=[
        'Poem ID',
        'Poem Name',
        'Verse',
        'Vazha',
        'Hewitt',
        'Google',
        'Gemini',
        'Gemini_ru',
        'ChatGPT',
        'ChatGPT_ru',
        'Hewitt - Google',
        'Hewitt - Gemini',
        'Hewitt - Gemini_ru',
        'Hewitt - ChatGPT',
        'Hewitt - ChatGPT_ru'
    ]
)

for poem in poem_names:
    poem_id = int(poem[:2])
    poem_name = poem[3:-3].replace('_', ' ').title()

    poem_vazha = [line.strip() for line in open(poems_path + 'ge/' + poem, "r")]
    poem_hewitt = [line.strip() for line in open(poems_path + 'en/' + poem, "r")]
    poem_google = [line.strip() for line in open(poems_path + 'ggl/' + poem, "r")]
    poem_gemini = [line.strip() for line in open(poems_path + 'gem/' + poem, "r")]
    poem_gemini_ru = [line.strip() for line in open(poems_path + 'gem_ru/' + poem, "r")]
    poem_chatgpt = [line.strip() for line in open(poems_path + 'gpt/' + poem, "r")]
    poem_chatgpt_ru = [line.strip() for line in open(poems_path + 'gpt_ru/' + poem, "r")]

    verses.extend(poem_hewitt)
    verses.extend(poem_google)
    verses.extend(poem_gemini)
    verses.extend(poem_gemini_ru)
    verses.extend(poem_chatgpt)
    verses.extend(poem_chatgpt_ru)

    poem_verse_counts.append(len(poem_vazha))
    for i in range(len(poem_vazha)):
        row = {
            'Poem ID': [poem_id],
            'Poem Name': [poem_name],
            'Verse': [i + 1],
            'Vazha': [poem_vazha[i]],
            'Hewitt': [poem_hewitt[i]],
            'Google': [poem_google[i]],
            'Gemini': [poem_gemini[i]],
            'Gemini_ru': [poem_gemini_ru[i]],
            'ChatGPT': [poem_chatgpt[i]],
            'ChatGPT_ru': [poem_chatgpt_ru[i]]
        }
        df_new = pd.DataFrame(row)
        df = pd.concat([df, df_new], ignore_index = True)

df.shape

(291, 15)

In [15]:
df.to_csv('results/all_texts_ru.csv')

In [16]:
verse_embeddings = model.encode(verses)
verse_embeddings.shape

(1746, 768)

In [17]:
for i, row in df.iterrows():
    poem_id = row['Poem ID']
    verse = row['Verse']
    offset = sum(poem_verse_counts[:poem_id-1]) * 6
    j = offset + verse - 1

    hewitt_google = cosine_similarity(
        [verse_embeddings[j]],
        [verse_embeddings[j + 1 * poem_verse_counts[poem_id - 1]]]
    )[0][0]
    
    hewitt_gemini = cosine_similarity(
        [verse_embeddings[j]],
        [verse_embeddings[j + 2 * poem_verse_counts[poem_id - 1]]]
    )[0][0]

    hewitt_gemini_ru = cosine_similarity(
        [verse_embeddings[j]],
        [verse_embeddings[j + 3 * poem_verse_counts[poem_id - 1]]]
    )[0][0]

    hewitt_chatgpt = cosine_similarity(
        [verse_embeddings[j]],
        [verse_embeddings[j + 4 * poem_verse_counts[poem_id - 1]]]
    )[0][0]

    hewitt_chatgpt_ru = cosine_similarity(
        [verse_embeddings[j]],
        [verse_embeddings[j + 5 * poem_verse_counts[poem_id - 1]]]
    )[0][0]    

    df.at[i,'Hewitt - Google'] = hewitt_google
    df.at[i,'Hewitt - Gemini'] = hewitt_gemini
    df.at[i,'Hewitt - Gemini_ru'] = hewitt_gemini_ru
    df.at[i,'Hewitt - ChatGPT'] = hewitt_chatgpt
    df.at[i,'Hewitt - ChatGPT_ru'] = hewitt_chatgpt_ru

In [18]:
df.head()

,Poem ID,Poem Name,Verse,Vazha,Hewitt,Google,Gemini,Gemini_ru,ChatGPT,ChatGPT_ru,Hewitt - Google,Hewitt - Gemini,Hewitt - Gemini_ru,Hewitt - ChatGPT,Hewitt - ChatGPT_ru
0,1,I Believe I Always Have Believed,1,"მრწამს, მარად მიწამებია მუდმივ სიცოცხლე სულისა, კარგისა, ქვეყნის მოყვარის, ქვეყნის ბედისგან წყლულისა.","I believe, I always have believed In the eternal life of the soul, The good, the lover of the world, Scarred by the fate of this same world.","I believe, there are always lands eternal life of the soul, Good, country lover, Ulcer from the fate of the country.","I believe, I've always believed In the eternal life of the soul, Of the good, the lover of the country, The one pained by the country's fate.","I believe, I have always believed in the eternal life of the soul, in goodness, in love for my homeland, in the pain of its fate.","I believe, I have always believed in the eternal life of the soul, of the good, the lover of the homeland, the one suffering from the fate of the country.","I believe, I have always believed in the eternal life of the soul, a kind soul, loving its country, suffering from the fate of the country.",0.534823,0.741352,0.799305,0.767628,0.783258
1,1,I Believe I Always Have Believed,2,"მრწამს, ფერფლნი კარგის გულისა ქარმ რო გაფანტოს ხმელადა, თვითოში მაინც ენთება ტიალ-სურვილი ცხელადა, –","I believe that, though the ashes of kind hearts Be scattered over the land by the wind, Yet in each of us there will burn Hotly a wish forlorn —","I believe that ashes are the heart of a good person Let the wind scatter the land, It still burns in itself Tial-desire was hot, –","I believe, even when the ashes Of a good heart are scattered by the wind, Within each one, a spark still ignites, A burning desire for warmth.","I believe that even if the ashes of a kind heart are scattered by the wind across the earth, within it still burns a fiery desire,","I believe, that even if the ashes of a good heart are scattered by the wind, they will still ignite a burning desire within, —","I believe that, when the wind scatters the ashes of a kind heart across the earth, there will still burn a fervent desire for emptiness, –",0.731435,0.758544,0.784163,0.825588,0.781224
2,1,I Believe I Always Have Believed,3,"ავის მჩაგრავად, კეთილის მუდამ იქნება მცველადა; ბეჩავის, გაჭირებულის მომხმარედ, მეშვლად, მხსნელადა.","To repress evil, and that this Will ever be a defence of the good, Help, succour and saviour Of the downtrodden and needy.","Evil oppressor, good There will always be a guard; Bechav, needy He helped me, helped me, saved me.","To suppress evil, to protect the good, It will always be a guardian; For the poor, the troubled, A helper, a savior, a rescuer.","For good, oppressed by evil, to always be a defender; for the poor and needy, a helper, deliverer, savior.","as the protector against evil, the good will always be a guardian; for the helpless, the downtrodden a helper, a supporter, a savior.","It will be an oppressor of evil, always a defender of good; it will support, protect, and save the poor and the needy.",0.419882,0.725808,0.782944,0.590861,0.542135
3,1,I Believe I Always Have Believed,4,"კარგს გულს არა ჰკლავს ბუნება, თან დააქვს ძველის-ძველადა!","Nature does not totally extinguish a kind heart, She carries it along with her from the very dawn of time.",Nature does not kill a good heart Brings old things with him!,"Nature does not kill a good heart, It carries it along, from time immemorial!","A kind heart is not killed by nature, it carries it with itself eternally!","Nature does not kill a good heart, it carries the wisdom of the ancients!","Nature does not kill a kind heart, it has existed since ancient times!",0.661742,0.806899,0.824055,0.761201,0.84506
4,2,Bakur,1,"– გვიამბე, ელიზბარაო, რა ამბავ მოხდა ეღოსა?","Tell us, Elizbar, In Egho what did occur.","– Tell us, Elizabeth What happened to Egosa?","– Tell us, Elizbara, What happened at dawn?","– Tell us, Elizbarao, what happened in Egosi?","Tell us, Elizbar, wh

In [19]:
df.to_csv('results/poems/similarity/cosine_similarity_ru.csv')

In [20]:
df_mean_hewitt_google = df.groupby('Poem Name')['Hewitt - Google'].mean().reset_index()
df_mean_hewitt_gemini = df.groupby('Poem Name')['Hewitt - Gemini'].mean().reset_index()
df_mean_hewitt_gemini_ru = df.groupby('Poem Name')['Hewitt - Gemini_ru'].mean().reset_index()
df_mean_hewitt_chatgpt = df.groupby('Poem Name')['Hewitt - ChatGPT'].mean().reset_index()
df_mean_hewitt_chatgpt_ru = df.groupby('Poem Name')['Hewitt - ChatGPT_ru'].mean().reset_index()

df_std_hewitt_google = df.groupby('Poem Name')['Hewitt - Google'].std().reset_index()
df_std_hewitt_gemini = df.groupby('Poem Name')['Hewitt - Gemini'].std().reset_index()
df_std_hewitt_gemini_ru = df.groupby('Poem Name')['Hewitt - Gemini_ru'].std().reset_index()
df_std_hewitt_chatgpt = df.groupby('Poem Name')['Hewitt - ChatGPT'].std().reset_index()
df_std_hewitt_chatgpt_ru = df.groupby('Poem Name')['Hewitt - ChatGPT_ru'].std().reset_index()

In [21]:
print(df_mean_hewitt_google)

# Poem-Wise Cosine Similarity Scores
df_mean_hewitt_google.shape
table = pd.concat([df_mean_hewitt_google, df_std_hewitt_google, df_mean_hewitt_gemini, df_std_hewitt_gemini,
                  df_mean_hewitt_gemini_ru, df_std_hewitt_gemini_ru, df_mean_hewitt_chatgpt, df_std_hewitt_chatgpt,
                 df_mean_hewitt_chatgpt_ru, df_std_hewitt_chatgpt_ru], axis = 1)
table.drop(columns = ["Poem Name"], inplace = True)
table = pd.concat([df_mean_hewitt_google["Poem Name"], table], axis = 1)
table.index = table.index + 1

#print(table)

# Average Cosine Similarity Scores
print('Google', df_mean_hewitt_google['Hewitt - Google'].mean(), df_std_hewitt_google['Hewitt - Google'].mean())
print('Gemini', df_mean_hewitt_gemini['Hewitt - Gemini'].mean(), df_std_hewitt_gemini['Hewitt - Gemini'].mean())
print('Gemini_ru', df_mean_hewitt_gemini_ru['Hewitt - Gemini_ru'].mean(), df_std_hewitt_gemini_ru['Hewitt - Gemini_ru'].mean())
print('ChatGPT', df_mean_hewitt_chatgpt['Hewitt - ChatGPT'].mean(), df_std_hewitt_chatgpt['Hewitt - ChatGPT'].mean())
print('ChatGPT_ru', df_mean_hewitt_chatgpt_ru['Hewitt - ChatGPT_ru'].mean(), df_std_hewitt_chatgpt_ru['Hewitt - ChatGPT_ru'].mean())

                                 Poem Name Hewitt - Google
0                                   Amiran        0.608679
1   As Once You Did O Lady As Once You Did         0.64519
2                                    Bakur        0.537513
3                         Consciences Song        0.546356
4         I Believe I Always Have Believed         0.58697
5     I Feel Like Singing And I Shall Sing         0.67313
6                               Loneliness        0.651978
7      Some Day It Will Happen I Shall Die        0.526339
8           That In Truth Is Not Manliness        0.375967
9             The Law Of The World Is Thus        0.593416
10               The Old Song Of Cavaliers        0.611858
11       Thrush Its The Same Song You Sing        0.566173
12                    Voice From The Grave        0.621796
13           What Created Me A Human Being        0.486629
14        Yet Again Shall I See The Spring          0.6008
Google 0.5755197080229463 0.1540665283837718
Gemini 0.71

In [111]:
max_cos_sim_hew_ggl = df['Hewitt - Google'].max()
max_cos_sim_hew_gem = df['Hewitt - Gemini'].max()
max_cos_sim_hew_gem_ru = df['Hewitt - Gemini_ru'].max()
max_cos_sim_hew_gpt = df['Hewitt - ChatGPT'].max()
max_cos_sim_hew_gpt_ru = df['Hewitt - ChatGPT_ru'].max()

min_cos_sim_hew_ggl = df['Hewitt - Google'].min()
min_cos_sim_hew_gem = df['Hewitt - Gemini'].min()
min_cos_sim_hew_gem_ru = df['Hewitt - Gemini_ru'].min()
min_cos_sim_hew_gpt = df['Hewitt - ChatGPT'].min()
min_cos_sim_hew_gpt_ru = df['Hewitt - ChatGPT_ru'].min()

print('Google', max_cos_sim_hew_ggl, min_cos_sim_hew_ggl)
print('Gemini', max_cos_sim_hew_gem, min_cos_sim_hew_gem)
print('Gemini_ru', max_cos_sim_hew_gem_ru, min_cos_sim_hew_gem_ru)
print('ChatGPT', max_cos_sim_hew_gpt, min_cos_sim_hew_gpt)
print('ChatGPT_ru', max_cos_sim_hew_gpt_ru, min_cos_sim_hew_gpt_ru)

Google 0.9659382 0.13076733
Gemini 1.0 0.3124894
Gemini_ru 0.94404775 0.19736117
ChatGPT 0.9643711 0.28665704
ChatGPT_ru 0.955553 0.28698295


In [112]:
df['Sum CS'] = df['Hewitt - Google'] + df['Hewitt - Gemini'] + df['Hewitt - Gemini_ru'] + df['Hewitt - ChatGPT'] + df['Hewitt - ChatGPT_ru']

#Most, Least Semantically Similar Verses
min_sum_cs = df.sort_values(by = 'Sum CS').head(3)
max_sum_cs = df.sort_values(by = 'Sum CS', ascending=False).iloc[:3, ]
print(min_sum_cs)
print(max_sum_cs)

    Poem ID                          Poem Name Verse  \
89        6     That In Truth Is Not Manliness     9   
12        2                              Bakur     9   
265      15  Thrush Its The Same Song You Sing     9   

                                                   Vazha  \
89   მგოსნობას არვის აცლიდე, მუდამ ჰკვეხდე და სცხარობდე.   
12            ბაკურის ომი და ქცევა ნეტავი თვალით გეხილა!   
265         მოგშივა? – არას იდარდებ, გაიკვებები მზაზედა.   

                                                                         Hewitt  \
89   When from your minstrelsy you give none peace But ever vaunt and overheat.   
12         Bak’ur’s skill in the fight You should have seen with your own eyes!   
265            Hungry? You will not grieve But feed on what you’ve stored away.   

                                                                Google  \
89               Don't let anyone miss you. Always boast and be proud.   
12   Bakury's war and conduct I wish I could see you 

In [29]:
kw_model = KeyBERT(model = 'all-mpnet-base-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [42]:
####### METHODOLOGY FOR OBTAINING KEYWORDS

# We encode verses using the MPNet-base model to compute the verse by verse semantic similarity. 
# We use the MPNet-base model for extraction of keywords (using KeyBERT) from all chapters. 
# However, given the constraint in the MPNet-base model that number of tokens should not exceed 384, it would not be possible to encode large chapters directly. 
# Hence, we propose a method to overcome this limitation by breaking each chapter into paragraphs of 15 verses. 
# We include 3 verses from the previous paragraph into the current paragraph to retain some context and maintain continuity. 
# For example, in the first paragraph, verses 1-15 are included, and in the second paragraph verses 13-27, then 25-39, and so on.  
# We keep the top 20 keywords because keywords that have a lower similarity score in the original paragraph may be more relevant when the entire paragraph is considered.
 
# Next, we extract the keywords for all paragraphs i with 20 candidate keywords of paragraph j such that i!=j. 
# For each keyword, we add up its cosine similarity score across paragraphs. 
# Finally, we obtain the top 10 keywords having the highest cumulative scores. 
# The key idea here is that if a term is a keyword in a certain paragraph, it also needs to be sufficiently close to other paragraphs in the higher dimensional vector space
# to qualify as a keyword for the entire chapter. 
# We use MMR with a diversity value of 0.5 to prevent the selection of similar meaning keywords.